# Stage 3 · RAG over the AI Media Dataset — project skeleton

*HSLU Computational Language Technologies · capstone project · built on `anatoolbox`*

This notebook is the skeleton of your Stage 3 project. It runs from top to bottom as it is. At
every step it gives you a **baseline** and a way to **measure** it; going beyond the baselines is
your job.

It follows Chapter 7 of *The Art of AI Product Development*, the recommended reading for Stage 3,
and adds what the project brief asks for on top:

| Part | Source | What you do |
|---|---|---|
| **A · Semantic search** | Chapter 7.2 | build the search index · evaluate retrieval · optimize retrieval |
| **B · End-to-end RAG** | Chapter 7.3 | set up answer generation · evaluate answers · optimize answers |
| **B4 · Knowledge graph** | Project brief | bring in your Stage 1 graph · compare with text-only RAG |
| **C · Results** | Project brief | compare all configurations · one limitation → one enhancement |

### How to read the cells

| Marker | Meaning |
|---|---|
| 🟦 **Baseline** | A standard method, ready to run — the reference point. |
| 📏 **Measure** | Evaluation of what ran before. Every result lands in a comparison table. |
| 🟩 **Worked example** | One optimization implemented completely, to show how it is done. |
| ✏️ **Your turn** | A template. It runs unchanged — reproducing the baseline — until you change it. |

Run the whole notebook once before changing anything: that gives you the baseline numbers your
changes are compared with.

### Team and disclosure

*Required for the submission — fill in before handing in.*

| Team member | Contribution |
|---|---|
| … | … |

**AI coding tools used:** *name each tool, and say how you used it.*

## 0 · Setup

### How this notebook uses `anatoolbox`

Every step calls a **tool** — `chunk_by_size`, `retrieve_passages`, `synthesize_answer`, … — and
passes its result on to the next tool as `input`. Every result carries `provenance`: which tool
made it, with which settings, from which earlier results. That is how each number in the final
tables traces back to the configuration that produced it.

The tools are baselines. To try your own method, there are three routes, from least to most code:

1. **Plug in a function** — e.g. your Stage 2 embedding model: `configure_embedder(my_model.encode)`.
2. **Subclass a tool and override one method** (a *hook*) — the one step you want to change.
   Everything else is inherited:

   ```python
   class ChunkBySentence(ChunkBySizeTool):
       tool_name = "chunk_by_sentence"      # a new name that keeps the prefix
       def split(self, text, settings):     # the one step you change
           ...
   ```
3. **Write a new tool** when no tool does the kind of work you need.

`show_hooks(SomeTool)` lists a tool's hooks; `docs/extending.md` in the anatoolbox repository
explains all three routes.

In [ ]:
# In Colab, install the toolbox first, as your course instructions describe, e.g.:
# %pip install -q "anatoolbox[embeddings]" pandas

import functools
import importlib.util
import inspect
import itertools
import json
import os
import platform
import re
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

import anatoolbox
from anatoolbox import ToolContext, resolve_tools

pd.set_option("display.max_colwidth", 120)
anatoolbox.register_reference_tools()
ctx = ToolContext()  # passed to every tool; a notebook needs nothing in it

(ingest, ingest_graph, chunk, rewrite, retrieve, rerank, synthesize,
 draft_questions, retrieval_metrics, judge) = resolve_tools([
    "ingest_corpus", "ingest_knowledge_graph", "chunk_by_size", "rewrite_query_for_retrieval",
    "retrieve_passages", "rerank_passages", "synthesize_answer",
    "extract_test_questions", "calculate_retrieval_metrics", "score_rag_answer",
])


def show_hooks(tool_class):
    # Print the methods a subclass of `tool_class` can override, with the first line of their docs.
    for name, member in vars(tool_class).items():
        if inspect.isfunction(member) and not name.startswith("_") and name not in ("run", "render"):
            doc = (inspect.getdoc(member) or "").split("\n")[0]
            print(f"  {name}{inspect.signature(member)}\n      {doc}")


print("python     :", platform.python_version())
print("anatoolbox :", anatoolbox.__version__)
has_models = importlib.util.find_spec("sentence_transformers") is not None
print("embeddings and reranking:", "available" if has_models else "install sentence-transformers")

### Project settings

Everything you are likely to change between runs is in this cell. The defaults are sized for a
full run on Colab; for a quick first run, lower `max_articles`, `qa_passages` and
`rag_eval_questions`.

In [ ]:
CONFIG = {
    "track": "Agentic Web",      # "Hardware & Infrastructure" | "Foundation Models" | "Agentic Web"
    "max_articles": None,        # None = the whole track; a number = an even sample over time
    "qa_passages": 120,          # passages to draft Q&A pairs from (the brief: ~100–150)
    "questions_per_passage": 2,  # 1–3, for ~200–300 Q&A pairs
    "rag_eval_questions": 50,    # questions per answer configuration in part B (2 LLM calls each)

    # Language models — any OpenAI-compatible endpoint. The brief requires a DIFFERENT model to
    # generate the Q&A pairs than the one in your RAG system; here that model also judges answers.
    "base_url": None,            # None = OpenAI; or e.g. "https://openrouter.ai/api/v1"
    "api_key": os.environ.get("OPENAI_API_KEY"),  # Colab: google.colab.userdata.get("OPENAI_API_KEY")
    "rag_model": None,           # ← e.g. "gpt-4o-mini": rewrites queries, writes answers
    "evaluation_model": None,    # ← a different, preferably stronger model: drafts Q&A, judges
}
# For automated test runs only: settings can be overridden without editing this cell.
CONFIG.update(json.loads(os.environ.get("COURSE_NOTEBOOK_OVERRIDES", "{}")))

DATA_DIR = Path(os.environ.get("AI_MEDIA_DATA_DIR", "data"))
RESULTS_DIR = DATA_DIR / "results" / re.sub(r"\W+", "_", CONFIG["track"].lower())
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps({key: value for key, value in CONFIG.items() if key != "api_key"}, indent=2))

In [ ]:
from anatoolbox.llm_client import call_llm_text, configure_llm, model_for

models = {"default": CONFIG["rag_model"], "evaluation": CONFIG["evaluation_model"]}
configure_llm(
    base_url=CONFIG["base_url"],
    api_key=CONFIG["api_key"],
    models={role: name for role, name in models.items() if name},
)


def check_model(role):
    # (is it usable?, a status line) for the model behind `role`.
    try:
        started = time.perf_counter()
        call_llm_text("Reply with one word.", "Say: ready", model=model_for(role), max_tokens=5)
        return True, f"{model_for(role)} — responding ({time.perf_counter() - started:.1f} s)"
    except Exception as error:  # no model configured, endpoint unreachable, wrong key, ...
        return False, f"not available — {type(error).__name__}: {str(error)[:150]}"


LLM_READY, rag_status = check_model("default")
EVAL_READY, eval_status = check_model("evaluation")
print("RAG model        :", rag_status)
print("evaluation model :", eval_status)
if LLM_READY and EVAL_READY and model_for("default") == model_for("evaluation"):
    print("\n⚠ The evaluation model is the RAG model: fine for a test run, not for your submission.")
if not LLM_READY:
    print("\nSet rag_model and evaluation_model in CONFIG. Cells that need a model are skipped until then.")

## 1 · Data: your track and your Stage 1 knowledge graph

The AI Media Dataset downloads from Kaggle's public API on the first run (~58 MB, no account
needed) and is cached in `data/`. Stage 3 works on the articles of **your track**. The keyword
patterns below are a rough stand-in: replace them with the track definition from your Stage 1 work.

In [ ]:
import io
import urllib.request
import zipfile

KAGGLE_URL = "https://www.kaggle.com/api/v1/datasets/download/jannalipenkova/ai-media-dataset"
TRACKS = {  # ← replace with your own track definition
    "Hardware & Infrastructure": r"\bgpus?\b|accelerator|nvidia|data cent(?:er|re)|semiconductor|\btsmc\b|\bchips?\b",
    "Foundation Models": r"foundation model|large language model|\bllms?\b|gpt-?\d|\bllama\b|gemini|claude|mistral|\bqwen\b",
    "Agentic Web": r"ai agents?|agentic|\bmcp\b|model context protocol|agent2agent|\ba2a\b|browser agent|computer use",
}


def dataset_csv():
    # The dataset CSV: $AI_MEDIA_CSV, a cached copy in DATA_DIR, or a fresh download.
    if os.environ.get("AI_MEDIA_CSV") and Path(os.environ["AI_MEDIA_CSV"]).exists():
        return Path(os.environ["AI_MEDIA_CSV"])
    cached = sorted(DATA_DIR.glob("ai_media_dataset_*.csv"))
    if cached:
        return cached[-1]
    print("Downloading the AI Media Dataset from Kaggle …")
    with urllib.request.urlopen(KAGGLE_URL, timeout=180) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))
    name = next(n for n in archive.namelist() if n.endswith(".csv"))
    archive.extract(name, DATA_DIR)
    return DATA_DIR / name


df = pd.read_csv(dataset_csv()).rename(columns={"Unnamed: 0": "id"})
searchable = (df["title"] + " " + df["content"] + " " + df["tags"]).str.lower()
track_df = df[searchable.str.contains(TRACKS[CONFIG["track"]], regex=True)].sort_values("date")
if CONFIG["max_articles"] and len(track_df) > CONFIG["max_articles"]:
    step = len(track_df) / CONFIG["max_articles"]  # an even sample over time
    track_df = track_df.iloc[[int(i * step) for i in range(CONFIG["max_articles"])]]
track_csv = RESULTS_DIR / "track_articles.csv"
track_df.to_csv(track_csv, index=False)

articles = ingest.run({"path": str(track_csv), "name": "track_articles", "text_field": "content"}, context=ctx)
print(f"{CONFIG['track']}: {articles['records']:,} articles, {track_df['date'].min()} → {track_df['date'].max()}")

### Your Stage 1 knowledge graph

Stage 1 ends with a knowledge graph and its schema, stored for reuse. `ingest_knowledge_graph`
loads them from one of two formats:

- an **edge table** (CSV, TSV, JSON or JSONL) with one row per fact: `subject`, `relation`,
  `object`, and optionally `subject_type`, `object_type`, `source_ids` (the ids of the articles the
  fact comes from) and `date`. Columns named differently can be mapped with `subject_field=`,
  `relation_field=` and `object_field=`;
- **NetworkX node-link JSON**, as written by `json.dump(networkx.node_link_data(G), file)`.

Keep `source_ids`: they make graph evidence traceable to articles, which the brief requires.

**Until your file is in place, a placeholder graph is used**: co-mentions of a handful of
hand-picked names. It only exists so the notebook runs end to end. It is *not* a Stage 1 graph, and
results with it say nothing about graph-enhanced RAG.

In [ ]:
from anatoolbox.graph import KnowledgeGraph, get_graph, register_graph

GRAPH_FILE = DATA_DIR / "stage1_graph_edges.csv"   # ← your Stage 1 graph
SCHEMA_FILE = DATA_DIR / "stage1_schema.md"         # ← your schema / data dictionary

PLACEHOLDER_NAMES = {
    "Agentic Web": ["OpenAI", "Anthropic", "Google", "Microsoft", "Amazon", "Salesforce", "Perplexity",
                    "MCP", "A2A", "Operator", "Claude", "Gemini", "Copilot", "Agentforce"],
    "Hardware & Infrastructure": ["Nvidia", "AMD", "Intel", "TSMC", "Broadcom", "Google", "Microsoft",
                                  "Amazon", "Meta", "Blackwell", "H100", "H20", "CoreWeave", "OpenAI"],
    "Foundation Models": ["OpenAI", "Anthropic", "Google", "Meta", "Mistral", "DeepSeek", "Alibaba",
                          "GPT-4o", "GPT-5", "Claude", "Gemini", "Llama", "Qwen", "Grok"],
}


def placeholder_graph(articles_df, names, min_articles=3):
    # Pairs of names mentioned in the same articles. A stand-in, NOT a Stage 1 graph.
    articles_by_pair = {}
    for article_id, title, content in articles_df[["id", "title", "content"]].itertuples(index=False):
        text = f"{title} {content}"
        mentioned = sorted(n for n in names if re.search(rf"(?<!\w){re.escape(n)}(?!\w)", text))
        for pair in itertools.combinations(mentioned, 2):
            articles_by_pair.setdefault(pair, []).append(str(article_id))
    facts = [
        {"subject": a, "relation": "co_mentioned_with", "object": b, "weight": len(ids), "source_ids": ids[:20]}
        for (a, b), ids in articles_by_pair.items()
        if len(ids) >= min_articles
    ]
    return register_graph(KnowledgeGraph.from_records(facts, name="placeholder_graph"))


if GRAPH_FILE.exists():
    schema = {"schema_path": str(SCHEMA_FILE)} if SCHEMA_FILE.exists() else {}
    GRAPH = get_graph(ingest_graph.run({"path": str(GRAPH_FILE), **schema}, context=ctx)["graph"])
    GRAPH_IS_PLACEHOLDER = False
else:
    GRAPH = placeholder_graph(track_df, PLACEHOLDER_NAMES[CONFIG["track"]])
    GRAPH_IS_PLACEHOLDER = True
    print(f"⚠ No {GRAPH_FILE.name} in the data folder — using the placeholder graph.\n")

summary = GRAPH.describe()
print(f"{summary['facts']:,} facts about {summary['entities']:,} entities · "
      f"{summary['facts_with_sources']:,} with source articles · {summary['dated_facts']:,} dated")
print("relations:", dict(list(summary["relations"].items())[:8]))

---
# Part A · Semantic search (Chapter 7.2)

A passage that is not retrieved cannot be used in an answer. Part A builds the search index,
evaluates it, and optimizes it — in that order, so every optimization is measured against the
baseline.

## A1 · Searching with semantic embeddings (7.2.2)

### 🟦 Baseline: fixed-size chunks

Embedding models work best on texts of a moderate, uniform length, and articles are long and vary
a lot. `chunk_by_size` cuts every article into windows of `size` words that overlap by `overlap`
words — ignoring sentences, paragraphs and meaning. That is what makes it a baseline.

In [ ]:
from anatoolbox.corpus import get_corpus

chunks = chunk.run({"input": articles, "size": 200, "overlap": 40}, context=ctx)
print(f"{chunks['chunks']:,} chunks from {chunks['articles']:,} articles · words per chunk: {chunks['tokens']}")
display(pd.DataFrame(get_corpus(chunks["corpus"]).records[:3])[["id", "title", "date", "tokens", "chunk_text"]])

### 🟦 Baseline: the embedding database and semantic search

Dense retrieval embeds every chunk once, embeds the question with the same model, and returns the
`size` chunks most similar to it — the chapter's *top k*. Too small a *k* misses evidence; too
large a *k* adds noise and cost further down the pipeline. The baseline model is `all-MiniLM-L6-v2`.

Here the embeddings are kept in memory, which is enough for one track. Larger collections need a
vector database (the chapter uses Weaviate; FAISS, Milvus and pgvector are alternatives).

`search(...)` below is the one retrieval call the rest of the notebook uses.

In [ ]:
from anatoolbox.corpus import configure_embedder

configure_embedder(None)  # None = the baseline model, all-MiniLM-L6-v2


def search(question, corpus=None, strategy="dense", size=30, tool=None, **args):
    # Retrieve `size` passages for `question` from a chunk corpus (default: the baseline chunks).
    tool = tool or retrieve
    request = {"query": question, "input": corpus or chunks, "strategy": strategy, "size": size, **args}
    return tool.run(request, context=ctx)


SAMPLE_QUESTIONS = {  # ← questions of your own, for looking at results by eye
    "Agentic Web": ["What is the Model Context Protocol, and who supports it?",
                    "What security risks do AI browser agents create?"],
    "Hardware & Infrastructure": ["How are export controls affecting Nvidia's sales in China?",
                                  "Which companies are building their own AI chips?"],
    "Foundation Models": ["Which open-weight models did DeepSeek release?",
                          "How do reasoning models differ from earlier LLMs?"],
}[CONFIG["track"]]

started = time.perf_counter()
example = search(SAMPLE_QUESTIONS[0], size=5)
print(f"embedded {chunks['chunks']:,} chunks and searched in {time.perf_counter() - started:.0f} s")
display(pd.DataFrame(example["passages"])[["rank", "score", "date", "title", "snippet"]])

## A2 · Evaluating search (7.2.3)

The chapter builds up evaluation in three steps: qualitative evaluation, quantitative evaluation,
and monitoring once the system is in use.

### Qualitative evaluation

Qualitative evaluation means people judging results: user studies, task-based evaluation, and
relevance assessments. The first two need users; a relevance assessment you can do right here. For
each sample question: do the top results answer it? Where keyword search (sparse) and semantic
search (dense) disagree, which one is right, and why?

In [ ]:
for question in SAMPLE_QUESTIONS:
    top5 = {strategy: [f"{p['date']} · {str(p['title'])[:70]}" for p in search(question, strategy=strategy, size=5)["passages"]]
            for strategy in ("sparse", "dense")}
    display(Markdown(f"**{question}**"))
    display(pd.DataFrame(top5))

### A test set: draft, review, reuse

Quantitative evaluation needs *ground truth*: questions whose answers, and source articles, are
known. Following the brief, the **evaluation model** drafts Q&A pairs from passages of your track,
mixing factual, temporal, relational, analytical and comparative questions. Each pair keeps the
article it came from (`source_ids`), the passage (`passage_id`) and a reference answer.

**The draft is not your test set yet.** The brief requires a manual review:

1. Open `qa_draft.json` in the results folder.
2. Fix unclear questions and wrong answers; delete unanswerable, trivial and duplicate pairs; add
   questions of your own, with the `source_ids` of the articles that answer them.
3. Save the result as `qa_reviewed.json` in the same folder. From then on, this cell loads it.

✏️ **Sampling.** The baseline samples passages at random. The brief asks for passages that cover
different subtopics, entities and time periods — override `sample` to get that coverage.

**Keep in mind:** generated questions tend to reuse the passage's own words, which favours keyword
search. And other articles may answer a question too, so retrieving an article other than the
source is not necessarily a mistake.

In [ ]:
from anatoolbox.extract.extract.extract_test_questions import ExtractTestQuestionsTool


class MyQuestionDrafter(ExtractTestQuestionsTool):
    tool_name = "extract_test_questions_with_my_sampling"   # ← name your method

    def sample(self, corpus, settings):
        # ✏️ YOUR TURN: the passages (corpus records) to draft questions from, e.g. stratified by
        # month, so every period is covered. settings["sample_size"] is how many to return.
        return super().sample(corpus, settings)   # ← a random sample, until you replace it


DRAFT_FILE = RESULTS_DIR / "qa_draft.json"
REVIEWED_FILE = RESULTS_DIR / "qa_reviewed.json"

if REVIEWED_FILE.exists():
    EVAL_SET = json.loads(REVIEWED_FILE.read_text())
    print(f"loaded {len(EVAL_SET)} reviewed questions from {REVIEWED_FILE.name}")
elif EVAL_READY:
    started = time.perf_counter()
    drafted = MyQuestionDrafter().run({
        "input": chunks, "sample_size": CONFIG["qa_passages"],
        "questions_per_record": CONFIG["questions_per_passage"], "min_chars": 400, "seed": 42,
    }, context=ctx)
    EVAL_SET = drafted["questions"]
    DRAFT_FILE.write_text(json.dumps(EVAL_SET, indent=2, ensure_ascii=False))
    print(f"drafted {len(EVAL_SET)} questions from {drafted['records_sampled']} passages "
          f"with {drafted['model']} in {time.perf_counter() - started:.0f} s → {DRAFT_FILE.name}")
    print("⚠ Not reviewed yet: review it and save qa_reviewed.json before you report results.")
else:
    EVAL_SET = []
    print("No evaluation model and no reviewed test set: the evaluation cells are skipped.")

if EVAL_SET:
    print("question types:", pd.Series([q.get("type") or "untyped" for q in EVAL_SET]).value_counts().to_dict())
    display(pd.DataFrame(EVAL_SET)[["id", "type", "question", "reference_answer", "date"]].head(8))

### 📏 Quantitative evaluation

For each test question, the article it came from should be retrieved. `calculate_retrieval_metrics`
reports the chapter's metrics, at several cut-offs *k*:

- **precision@k** — the share of the top *k* articles that are relevant: is retrieval returning noise?
- **recall@k** — the share of the relevant articles in the top *k*: is it missing evidence?
- **MRR** — the mean of 1 / rank of the first relevant article: does relevant evidence come first?
- **hit@k** — whether any relevant article is in the top *k*. With one source article per question,
  it equals recall@k.

Several chunks of one article count as that article, once. `evaluate_retrieval(label, search_fn)`
runs a search function over the whole test set and adds a row to the comparison table; every
optimization below uses it.

In [ ]:
RETRIEVAL_RESULTS = []   # one row per configuration → the comparison tables
RETRIEVAL_METRICS = {}   # configuration → the full metrics, including every question
K = [1, 3, 5, 10]


def evaluate_retrieval(label, search_fn):
    # Run `search_fn(question)` for every test question, score the results, add a table row.
    if not EVAL_SET:
        print(f"skipped {label!r}: no test set")
        return
    started = time.perf_counter()
    runs = [{"id": q["id"], "question": q["question"], "relevant": q["source_ids"], "retrieved": search_fn(q["question"])}
            for q in EVAL_SET]
    metrics = retrieval_metrics.run({"results": runs, "k": K}, context=ctx)
    provenance = runs[-1]["retrieved"]["provenance"]
    settings = {key: value for key, value in provenance["settings"].items()
                if key not in ("query", "queries") and value not in (None, [], {})}
    RETRIEVAL_METRICS[label] = metrics
    RETRIEVAL_RESULTS.append({"configuration": label, "tool": provenance["tool"], **metrics["summary"],
                              "seconds": round(time.perf_counter() - started, 1), "settings": settings})


def by_question_type(label, columns=("hit@1", "hit@5", "recall@10", "reciprocal_rank")):
    # Mean metrics per question type, for one evaluated configuration.
    types = {q["id"]: q.get("type") or "untyped" for q in EVAL_SET}
    rows = pd.DataFrame(RETRIEVAL_METRICS[label]["per_question"]).assign(type=lambda d: d["id"].map(types))
    return rows.groupby("type")[list(columns)].mean().round(3).assign(questions=rows.groupby("type").size())


evaluate_retrieval("🟦 dense (baseline)", search)
if RETRIEVAL_RESULTS:
    display(pd.DataFrame(RETRIEVAL_RESULTS))
    display(by_question_type("🟦 dense (baseline)"))

### Real-world monitoring

A test set is a snapshot, and real questions drift away from it. In use, a search system is watched
through **click-through rates** (are the top results the ones people open?), **query logs** (what do
people ask, and where do they give up?), **engagement** such as dwell time, and **session analysis**
(how many searches does one task take?). Failures become new test questions. For the project, it is
enough to describe how you would monitor your system.

## A3 · Optimizing your search system (7.2.4)

One block per optimization in the chapter, in the chapter's order. Each ends with an
`evaluate_retrieval(...)` call, so its numbers join the comparison table. You don't have to do all of
them — pick the ones your evaluation points to.

### ✏️ A3.1 · Advanced chunking methods

Fixed windows cut through sentences and mix topics. The chapter's alternatives:

- **sentence or paragraph chunking** — for texts whose parts stand on their own;
- **semantic chunking** — split where the meaning shifts, so each chunk keeps one topic;
- **hierarchical chunking** — several levels (article → paragraphs → sentences), for finding the
  right granularity;
- **chunk size** — short chunks retrieve specific details, long chunks keep context. This experiment
  needs no code: run the baseline again with `size=100` or `size=400`.

Contextual chunking — adding what a chunk lost, such as the article title — is prepared too: see
`contextualize=` and `register_contextualizer`. To chunk differently, override `split`:

In [ ]:
from anatoolbox.preprocess.chunk.chunk_by_size import ChunkBySizeTool

show_hooks(ChunkBySizeTool)

In [ ]:
class MyChunker(ChunkBySizeTool):
    tool_name = "chunk_by_my_method"   # ← name your method (keep the "chunk_" prefix)
    description = "…"                  # ← one sentence on what it does

    def split(self, text, settings):
        # ✏️ YOUR TURN: the chunks of one article, in order, as [{"text": ...}, ...].
        # settings holds the arguments (size, overlap, …); add your own in settings().
        return super().split(text, settings)   # ← the baseline, until you replace it


my_chunks = MyChunker().run({"input": articles, "size": 200, "overlap": 40}, context=ctx)
print(f"{my_chunks['chunks']:,} chunks · words per chunk: {my_chunks['tokens']}")
evaluate_retrieval("✏️ dense on MyChunker", lambda question: search(question, corpus=my_chunks))

### ✏️ A3.2 · Fine-tuning the embedding model

A general-purpose embedding model does not know your track's vocabulary — that "A2A" and
"Agent2Agent" are the same thing, or which terms belong together. Plug in the model you trained or
fine-tuned in Stage 2: any function from a list of texts to vectors works. Changing the embedding
model re-embeds the corpus.

In [ ]:
# from sentence_transformers import SentenceTransformer
# stage2_model = SentenceTransformer("path/to/your/stage2/model")
# configure_embedder(lambda texts: stage2_model.encode(texts, normalize_embeddings=True))
# evaluate_retrieval("✏️ dense with the Stage 2 model", search)
# configure_embedder(None)   # back to the baseline model for the rest of the notebook

### 🟩 A3.3 · Integrating lexical search for precision — worked example

Embeddings capture meaning, but they blur exact names, version numbers and acronyms — which news
questions are full of. Lexical (keyword) search with BM25 matches them literally. Three ways to
use it:

1. **sparse** — BM25 alone (shipped);
2. **hybrid** — reciprocal rank fusion, which merges the BM25 and dense *rankings* (shipped);
3. **weighted hybrid** — a weighted sum of the two *scores*, `α · dense + (1 − α) · BM25`, after
   scaling each to 0–1. Not shipped: it is built below as a subclass, showing the whole pattern —
   a new name, a new argument, and one overridden hook.

In [ ]:
evaluate_retrieval("🟦 sparse", lambda question: search(question, strategy="sparse"))
evaluate_retrieval("🟦 hybrid", lambda question: search(question, strategy="hybrid"))

In [ ]:
from anatoolbox.corpus import ensure_bm25, ensure_embeddings, get_embedder
from anatoolbox.gather.retrieve.retrieve_passages import RetrievePassagesTool
from anatoolbox.retrieval import Hit, dense_rank
from anatoolbox.tool import with_properties


def scaled(scores):
    # Min-max scale {index: score} to 0–1, so BM25 and cosine scores can be added.
    if not scores:
        return {}
    low, high = min(scores.values()), max(scores.values())
    return {i: (s - low) / (high - low) if high > low else 1.0 for i, s in scores.items()}


class RetrieveWeightedHybrid(RetrievePassagesTool):
    '''Hybrid search as a weighted sum of scaled BM25 and embedding scores.'''

    # 1 · A new name that keeps the prefix, and what the tool does.
    tool_name = "retrieve_passages_weighted_hybrid"
    description = "Rank passages by alpha * dense score + (1 - alpha) * BM25 score, both scaled to 0-1."
    strategies = ("weighted_hybrid",)

    # 2 · A new argument: declared in the input schema, read and checked in settings().
    input_schema = with_properties(RetrievePassagesTool.input_schema, {
        "alpha": {"type": "number", "minimum": 0, "maximum": 1, "description": "Weight of the dense score."},
    })

    def settings(self, args):
        alpha = float(args.get("alpha", 0.5))
        if not 0 <= alpha <= 1:
            raise self.input_error(f"alpha must be between 0 and 1, got {alpha}.", argument="alpha")
        return {**super().settings({**args, "strategy": "weighted_hybrid"}), "alpha": alpha}

    # 3 · The one step that changes: how one query ranks the corpus.
    def rank(self, query, corpus, settings, limit):
        bm25 = scaled({hit.index: hit.score for hit in ensure_bm25(corpus).rank(query)})
        dense = scaled({hit.index: hit.score for hit in dense_rank(get_embedder()([query]), ensure_embeddings(corpus))})
        alpha = settings["alpha"]
        combined = {i: alpha * dense.get(i, 0.0) + (1 - alpha) * bm25.get(i, 0.0) for i in bm25.keys() | dense.keys()}

        hits = []
        for index, score in sorted(combined.items(), key=lambda item: (-item[1], item[0])):
            # keep() is inherited, so date ranges and metadata filters still apply.
            # (A recency boost would need self.boost(...) here; this example leaves it out.)
            if self.keep(corpus.records[index], settings):
                hits.append(Hit(index=index, score=score, strategy="weighted_hybrid"))
                if len(hits) == limit:
                    break
        return hits


weighted_hybrid = RetrieveWeightedHybrid()
for alpha in (0.25, 0.5, 0.75):
    evaluate_retrieval(f"🟩 weighted hybrid α={alpha}",
                       lambda question, alpha=alpha: search(question, tool=weighted_hybrid, alpha=alpha))
if RETRIEVAL_RESULTS:
    display(pd.DataFrame(RETRIEVAL_RESULTS)[["configuration", "tool", "hit@1", "hit@5", "recall@10", "mrr", "settings"]])

**Reading the worked example.** Each row names the tool that produced it, and its settings record
`alpha`, so configurations cannot be mixed up. Two cautions for your own comparisons:

- **Choosing α on the test set and reporting the score from the same set overstates it.** Choose α
  on one part of your questions and report on the rest.
- **With a few dozen questions, small differences are noise.** Look at the per-question results in
  `RETRIEVAL_METRICS[label]["per_question"]` before you call a winner.

### ✏️ A3.4 · Leveraging metadata to refine search results

Every article has a publication date and a domain. The chapter uses metadata in two ways:
**filtering** (e.g. only the last six months, only certain sources) and **recency bias** (newer
articles rank higher). Shipped: `date_from` / `date_to`, `filters` and `recency_half_life_days` —
an article one half-life older scores half as much. Your own rules go into `keep` (which records may
be returned) and `boost` (a score multiplier).

In [ ]:
show_hooks(RetrievePassagesTool)

evaluate_retrieval("🟦 dense + recency (half-life 180 days)",
                   lambda question: search(question, recency_half_life_days=180))


class MyMetadataRetriever(RetrievePassagesTool):
    tool_name = "retrieve_passages_with_my_metadata_rules"   # ← name your method

    def keep(self, record, settings):
        # ✏️ YOUR TURN: e.g. exclude domains you don't trust, or keep the period a question names.
        return super().keep(record, settings)

    def boost(self, record, settings):
        # ✏️ YOUR TURN: e.g. favour recent articles for "latest" questions. 1.0 = unchanged.
        return super().boost(record, settings)


evaluate_retrieval("✏️ dense + MyMetadataRetriever", lambda question: search(question, tool=MyMetadataRetriever()))

### ✏️ A3.5 · Using reranking to address information loss

An embedding compresses a whole chunk into one vector, and details are lost on the way. A reranker
reads the question together with each candidate's full text and scores them again. It is slower
than embeddings and more accurate, so the chapter's recipe is: **embeddings for recall, reranking for
precision** — retrieve a generous set, then rerank it. Shipped: a pretrained cross-encoder. (The
chapter also trains rerankers on user clicks; you have no clicks, but you can try other scorers,
e.g. a language model judging relevance, by overriding `score`.)

Reranking below reorders all 30 retrieved passages, so its metrics compare fairly with retrieval
alone.

In [ ]:
from anatoolbox.gather.rerank.rerank_passages import RerankPassagesTool


def search_and_rerank(question, reranker=None):
    # Retrieve 30 passages, then let a reranker put them in a new order.
    return (reranker or rerank).run({"input": search(question), "keep": 30}, context=ctx)


evaluate_retrieval("🟦 dense → rerank (cross-encoder)", search_and_rerank)


class MyReranker(RerankPassagesTool):
    tool_name = "rerank_passages_with_my_scorer"   # ← name your method

    def score(self, query, texts, settings):
        # ✏️ YOUR TURN: one relevance score per text, higher = more relevant.
        return super().score(query, texts, settings)


# evaluate_retrieval("✏️ dense → MyReranker", lambda question: search_and_rerank(question, MyReranker()))

### 📏 Search: all configurations

Choose the retrieval configuration you carry into part B, and write down why.

In [ ]:
if RETRIEVAL_RESULTS:
    table = pd.DataFrame(RETRIEVAL_RESULTS).set_index("configuration")
    display(table[["tool", "hit@1", "hit@5", "recall@10", "precision@5", "mrr", "seconds"]].round(3))

---
# Part B · End-to-end RAG (Chapter 7.3)

## B1 · A basic RAG set-up (7.3.1)

The brief also points to Prof. Dr. Daniel Perruchoud's Cleantech-RAG tutorial for a text-based RAG
baseline; the baseline here plays the same role.

### Selecting a language model for response generation

Your RAG model is `CONFIG["rag_model"]`. The chapter's criteria for choosing it:

- it must handle **long prompts** full of retrieved passages — and input tokens are what you pay for;
- models use the **middle of a long prompt** poorly, so it matters how well yours does (the baseline
  puts the most relevant passages at the start and the end);
- should answers draw on the model's **general knowledge**, or only on your documents? That is the
  baseline's `mode`: `"grounded"` (only the sources) or `"blended"` (sources plus marked background).

### 🟦 Constructing a basic RAG prompt

The chapter's basic RAG prompt has three parts, and `synthesize_answer` builds each of them:

| Prompt part | In `synthesize_answer` |
|---|---|
| **System prompt** — role and rules | `system_prompt(settings)`, printed below |
| **Instruction** — task, format, audience | the question, plus optional `instructions=` |
| **Context** — retrieved passages with metadata | the passages as `[S1]…[Sn]` with title, date and URL |

The answer cites labels, and the tool checks the citations instead of trusting them: **unknown
citations** (labels never given), **uncited sources**, and **citation coverage** (the share of
sentences with a citation).

The baseline system: dense retrieval of 30 chunks → at most 6 passages, at most 2 per article →
answer. `answer(...)` below is the one generation call the rest of the notebook uses.

In [ ]:
from anatoolbox.enrich.synthesize.synthesize_answer import SynthesizeAnswerTool

ANSWER_ARGS = {"max_passages": 6, "max_per_source": 2, "max_chars_per_passage": 1200, "max_tokens": 400}


def answer(question, passages, tool=None, **args):
    # Answer `question` (a test-set entry, or {"question": ...}) from a retrieval or rerank result.
    tool = tool or synthesize
    return tool.run({"question": question["question"], "input": passages, **ANSWER_ARGS, **args}, context=ctx)


print(synthesize.system_prompt({"mode": "grounded"}))

if LLM_READY:
    started = time.perf_counter()
    example_answer = answer({"question": SAMPLE_QUESTIONS[0]}, search(SAMPLE_QUESTIONS[0]))
    display(Markdown(f"**{SAMPLE_QUESTIONS[0]}** · {example_answer['model']} · "
                     f"{time.perf_counter() - started:.0f} s\n\n{example_answer['answer_markdown']}"))
    print("cited:", example_answer["cited"], "· unknown:", example_answer["unknown_citations"],
          "· citation coverage:", example_answer["citation_coverage"])

## B2 · Evaluating your RAG system (7.3.2)

### Component-level evaluation

**Retrieval** was evaluated in part A. **Generation** is harder to evaluate: one question has many
valid answers. The chapter's approach is to hold retrieval fixed, so that differences come from
generation alone. Below, the passages for each test question are retrieved once, and every
generation variant in B3 answers from exactly those passages.

### 📏 End-to-end evaluation

The **evaluation model** acts as a judge. It reads each answer together with its sources and scores
it from 1 to 5, with a rationale, on:

- **faithfulness** (the chapter's *groundedness*) — is every statement supported by the sources?
- **relevance** (the chapter's *answer relevance*) — does the answer address the question, in the
  depth it asks for?
- **context grounding** — does it use the relevant passages and graph evidence, and cite them? (brief)
- **temporal grounding** — are dates and chronology right? Empty when the question is not about
  time. (brief)
- **correctness** — does it agree with the reference answer? A rough signal, since other answers
  can be valid too.

**A judge is a model too.** Your evaluation model drafted the reference answers and now judges
against them. `review_flags` marks judgments that contradict themselves; read those, and a sample of
the others, before you trust the averages.

In [ ]:
RAG_RESULTS = []   # one row per (configuration, question) → the comparison tables
RAG_SET = EVAL_SET[: CONFIG["rag_eval_questions"]]
CRITERIA = ["faithfulness", "relevance", "context_grounding", "temporal_grounding", "correctness"]

# Retrieval held fixed: the baseline passages for every question, retrieved once.
BASELINE_PASSAGES = {q["id"]: search(q["question"]) for q in RAG_SET}


def evaluate_rag(label, answer_fn):
    # Answer every question in RAG_SET with `answer_fn(question)`, judge each answer, add table rows.
    if not (RAG_SET and LLM_READY and EVAL_READY):
        print(f"skipped {label!r}: needs a test set, a RAG model and an evaluation model")
        return None
    started, rows = time.perf_counter(), []
    for q in RAG_SET:
        result = answer_fn(q)
        judgment = judge.run({"answer": result, "reference_answer": q["reference_answer"]}, context=ctx)
        rows.append({
            "configuration": label, "id": q["id"], "type": q.get("type") or "untyped", "question": q["question"],
            **judgment["scores"],
            "citation_coverage": result["citation_coverage"],
            "unknown_citations": len(result["unknown_citations"]),
            "graph_facts": len(result["facts"]),
            "review_flags": "; ".join(judgment["review_flags"]),
            "answer": result["answer"],
            "tool": result["provenance"]["tool"],
            "run": result["provenance"]["run_id"],
        })
    RAG_RESULTS.extend(rows)
    print(f"{label}: {len(rows)} answers judged in {time.perf_counter() - started:.0f} s")
    return pd.DataFrame(rows)


def show_rag(results):
    # Mean scores of one evaluate_rag(...) run, if it ran.
    if results is not None:
        display(results[CRITERIA + ["citation_coverage"]].mean().round(2).to_frame("mean").T)


BASELINE_RAG = "🟦 text-only RAG (baseline)"
baseline_results = evaluate_rag(BASELINE_RAG, lambda q: answer(q, BASELINE_PASSAGES[q["id"]]))
show_rag(baseline_results)
if baseline_results is not None:
    flagged = baseline_results[baseline_results["review_flags"] != ""]
    print(f"{len(flagged)} of {len(baseline_results)} judgments flagged for review")
    display(baseline_results[["id", "type", "question", "answer", "review_flags"]].head(5))

## B3 · Optimizing your RAG system (7.3.3)

### ✏️ B3.1 · Analyzing and enhancing the user query

Users ask vague, broad or compound questions. The chapter's techniques:

- **query expansion** — turn a broad question into several specific queries;
- **query transformation** — rewrite an unclear question into a clear, searchable one;
- **query classification** — recognize the kind of question and route it, e.g. temporal questions
  to date filters or a recency boost.

Shipped: `rewrite_query_for_retrieval` with `expand`, `decompose` (one query per part of a compound
question) and `clarify`; the rewrites are searched together and their rankings fused. A rewrite
changes retrieval, so it is measured twice: with the retrieval metrics, and end to end.

In [ ]:
from anatoolbox.gather.rewrite.rewrite_query_for_retrieval import RewriteQueryForRetrievalTool

show_hooks(RewriteQueryForRetrievalTool)


class MyQueryRewriter(RewriteQueryForRetrievalTool):
    tool_name = "rewrite_query_with_my_method"   # ← name your method

    def rewrite(self, settings, context):
        # ✏️ YOUR TURN: return {"queries": [...], "exact_terms": [...], "time_range": {...}}.
        # settings["question"] is the question; settings["strategy"] the requested strategy.
        return super().rewrite(settings, context)


query_rewriter = MyQueryRewriter()


def search_with_rewrites(question):
    # Rewrite the question, then search with the question and all its rewrites.
    rewrites = query_rewriter.run({"question": question, "strategy": "decompose"}, context=ctx)
    return search(question, queries_input=rewrites)


if LLM_READY:
    evaluate_retrieval("✏️ dense + query rewriting", search_with_rewrites)
    show_rag(evaluate_rag("✏️ query rewriting", lambda q: answer(q, search_with_rewrites(q["question"]))))

### ✏️ B3.2 · Optimizing the prompt

Retrieval stays fixed here, so the comparisons isolate the prompt. The chapter's techniques:

- **constraints that fit the content** — strict grounding for specific, factual answers; general
  knowledge allowed for exploratory ones. Shipped as `mode="grounded"` (the baseline) and
  `mode="blended"`. Expect blended answers to score lower on faithfulness — by design; the question is
  whether what they add is worth it;
- **chain-of-thought** — ask the model to work through the task in explicit steps;
- **reflection** — a second round in which the model reviews and improves its draft, at the cost of
  more calls.

In [ ]:
show_hooks(SynthesizeAnswerTool)

show_rag(evaluate_rag("🟦 blended mode", lambda q: answer(q, BASELINE_PASSAGES[q["id"]], mode="blended")))


class MyPromptedAnswer(SynthesizeAnswerTool):
    tool_name = "synthesize_answer_with_my_prompt"   # ← name your method

    def system_prompt(self, settings):
        # ✏️ YOUR TURN: replace or extend the instructions, e.g. with chain-of-thought steps.
        return super().system_prompt(settings)

    def generate(self, system, user, settings, context):
        # ✏️ Optional, e.g. reflection: draft = super().generate(...), then ask the model to check
        # the draft against the sources and return the improved version.
        return super().generate(system, user, settings, context)


my_prompted_answer = MyPromptedAnswer()
show_rag(evaluate_rag("✏️ my prompt", lambda q: answer(q, BASELINE_PASSAGES[q["id"]], tool=my_prompted_answer)))

### ✏️ B3.3 · Efficient augmentation and context curation

What reaches the prompt matters as much as what was retrieved. Shipped curation: duplicate passages
are dropped, the most relevant passages go to the start and the end of the prompt, and
`max_per_source` limits passages per article. The chapter's techniques beyond that:

- **fusion** — merge overlapping information from several sources into one statement, so the prompt
  is shorter and the answer does not repeat itself;
- **multi-turn retrieval** — retrieve, see what is still missing, retrieve again. That changes the
  pipeline rather than one step: write a function that calls `search` several times and answers from
  the combined passages.

To change which passages go into the prompt, and how, override `curate`:

In [ ]:
class MyCuratedAnswer(SynthesizeAnswerTool):
    tool_name = "synthesize_answer_with_my_curation"   # ← name your method

    def curate(self, passages, settings):
        # ✏️ YOUR TURN: choose, shorten, merge and order passages. super().curate(...) labels them
        # S1…Sn; keep those labels, the citation checks rely on them.
        return super().curate(passages, settings)


my_curated_answer = MyCuratedAnswer()
show_rag(evaluate_rag("✏️ my context curation", lambda q: answer(q, BASELINE_PASSAGES[q["id"]], tool=my_curated_answer)))

### B3.4 · Fine-tuning the LLM for domain-specific knowledge

The chapter fine-tunes a model on high-quality examples from the domain, so it writes with real
domain understanding and is less swayed by biased sources — and recommends it when prompt and query
optimizations are not enough, since it needs training data and compute. Facts that change over time
still belong in retrieval.

If you fine-tuned a compact model in Stage 2 (e.g. with LoRA), serve it through an
OpenAI-compatible server such as vLLM or Ollama, set `CONFIG["base_url"]` and `CONFIG["rag_model"]`,
and run part B again: the tables show whether it helps. Otherwise, discuss when it would be worth it.

## B4 · Integrating your knowledge graph (project brief)

The brief asks you to combine your Stage 1 graph with text retrieval, and to compare the result with
the text-only baseline on retrieval quality, answer quality, and temporal questions. The graph can
help in two places:

1. **as evidence in the prompt** — `answer(..., facts=[...])` shows graph facts as `[G1]…[Gn]`, cited
   and checked like passages; the judge sees them too;
2. **in retrieval** — e.g. boost passages from the articles behind the facts (`fact_source_ids`).

`graph_evidence` below is only a **starting point**: graph entities named literally in the question,
and the facts one hop around them. How to choose better evidence is your design — for example:

- **entity linking** — aliases, abbreviations, fuzzy matching (`find_entities` matches exact names);
- **relation selection** — use your schema to pick the relations that fit the question type;
- **time** — prefer facts dated near the period a question asks about.

Report how many questions get any graph evidence at all: without facts, a "graph-enhanced" answer is
just the text-only answer.

In [ ]:
from anatoolbox.graph import describe_fact, fact_source_ids


@functools.lru_cache(maxsize=None)
def graph_evidence(question, max_facts=15):
    # ✏️ YOUR TURN: the graph facts for `question`, most relevant first.
    entities = GRAPH.find_entities(question)
    return tuple(GRAPH.subgraph(entities, hops=1, max_facts=max_facts))


if EVAL_SET:
    with_evidence = sum(1 for q in EVAL_SET if graph_evidence(q["question"]))
    source = "the placeholder graph" if GRAPH_IS_PLACEHOLDER else "your Stage 1 graph"
    print(f"graph evidence for {with_evidence} of {len(EVAL_SET)} test questions, from {source}")
    for q in EVAL_SET:
        if graph_evidence(q["question"]):
            print("\nexample:", q["question"])
            for fact in graph_evidence(q["question"])[:5]:
                print("  ", describe_fact(fact), "· articles:", fact.get("source_ids", [])[:3])
            break


class MyGraphRetriever(RetrievePassagesTool):
    tool_name = "retrieve_passages_with_graph"   # ← name your method

    def boost(self, record, settings):
        # ✏️ YOUR TURN: e.g. favour passages from the articles behind the question's graph facts:
        #     articles = set(fact_source_ids(graph_evidence(settings["query"])))
        #     return 1.5 if record.get("source_id") in articles else 1.0
        return super().boost(record, settings)


GRAPH_RAG = "✏️ graph-enhanced RAG"
evaluate_retrieval("✏️ dense + MyGraphRetriever", lambda question: search(question, tool=MyGraphRetriever()))
show_rag(evaluate_rag(GRAPH_RAG, lambda q: answer(q, BASELINE_PASSAGES[q["id"]], facts=list(graph_evidence(q["question"])))))

---
# Part C · Results

### 📏 Retrieval: all configurations

In [ ]:
if RETRIEVAL_RESULTS:
    retrieval_table = pd.DataFrame(RETRIEVAL_RESULTS).set_index("configuration")
    display(retrieval_table[["tool", "hit@1", "hit@5", "recall@10", "precision@5", "mrr"]].round(3))
    retrieval_table.to_csv(RESULTS_DIR / "retrieval_results.csv")

### 📏 Answers: all configurations, per question type, and text-only vs graph-enhanced

The brief asks for results across **question categories**, and for **text-only RAG and RAG with your
Stage 1 graph** side by side.

In [ ]:
if RAG_RESULTS:
    rag = pd.DataFrame(RAG_RESULTS)
    rag.to_csv(RESULTS_DIR / "rag_results.csv", index=False)

    print("Mean scores per configuration")
    display(rag.groupby("configuration", sort=False)[CRITERIA + ["citation_coverage", "graph_facts"]].mean().round(2))

    print("Mean scores per question type and configuration")
    display(rag.groupby(["type", "configuration"], sort=False)[CRITERIA].mean().round(2))

    print("Text-only vs graph-enhanced RAG" + (" — placeholder graph, not meaningful" if GRAPH_IS_PLACEHOLDER else ""))
    compared = rag[rag["configuration"].isin([BASELINE_RAG, GRAPH_RAG])]
    display(compared.groupby(["type", "configuration"], sort=False)[CRITERIA + ["graph_facts"]].mean().round(2))

    print(f"{(rag['review_flags'] != '').sum()} of {len(rag)} judgments carry review flags — read them before reporting.")

### One limitation → one enhancement

The brief asks you to identify one limitation from your evaluation and to implement one enhancement.

- **Limitation observed:** *which configuration, which question types, which criterion — with numbers from the tables above.*
- **Likely cause:** *what in the pipeline produces it — look at the answers and their provenance.*
- **Enhancement:** *what you changed (which block, which subclass) and why it should help.*
- **Result:** *before and after, on the same questions and with the same judge model.*

In [ ]:
# Everything needed to trace a number in the tables back to what produced it.
provenance = {
    "articles": articles["provenance"],
    "chunks": chunks["provenance"],
    "retrieval": RETRIEVAL_RESULTS,
    "answers": [{key: row[key] for key in ("configuration", "id", "tool", "run")} for row in RAG_RESULTS],
}
(RESULTS_DIR / "provenance.json").write_text(json.dumps(provenance, indent=2, default=str))
print("saved in the results folder:", sorted(path.name for path in RESULTS_DIR.iterdir()))